<a href="https://colab.research.google.com/github/s68452093-code/bhashasetu/blob/main/bhashasetu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Mount Google Drive
Run the cell below to authenticate and mount your Google Drive to this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Copy the folder to Colab workspace
We will search for your folder `scout file imagies` in your Drive and copy it to `/content/scout_file_images`.

In [ ]:
import os
import shutil

# Source path inside Google Drive
# Note: If the folder is inside another directory, update this path accordingly
source_folder = '/content/drive/MyDrive/scout file imagies'
destination_folder = '/content/scout_file_images'

if os.path.exists(source_folder):
    shutil.copytree(source_folder, destination_folder, dirs_exist_ok=True)
    print(f"Successfully copied '{source_folder}' to '{destination_folder}'")
    print("Files inside the folder:")
    print(os.listdir(destination_folder))
else:
    print(f"Error: Folder '{source_folder}' not found. Please verify the folder name in your Google Drive.")

### Inspect Project Directory
Let's list the root files and see what files are present from the original project and the copied `scout_file_images` folder to plan our next steps.

In [1]:
import os

def list_files(startpath):
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files[:15]: # Show up to 15 files per directory to keep it clean
            print(f'{subindent}{f}')
        if len(files) > 15:
            print(f'{subindent}... and {len(files) - 15} more files')

print("=== Current Workspace Files ===")
list_files('/content')

=== Current Workspace Files ===
content/
    .config/
        default_configs.db
        gce
        .last_update_check.json
        hidden_gcloud_config_universe_descriptor_data_cache_configs.db
        config_sentinel
        active_config
        .last_survey_prompt.yaml
        .last_opt_in_prompt.yaml
        logs/
            2026.10.26/
                22.18.15.518759.log
                22.18.38.571729.log
                22.18.53.588181.log
                22.18.45.880512.log
                22.18.52.867204.log
                22.18.44.768076.log
        configurations/
            config_default
    drive/
        .shortcut-targets-by-id/
        MyDrive/
            checking regional.py
            collect_data.py
            isl_alphabet_model_v2.pkl
            isl_common.py
            train_model.py
            training_report.txt
            translate3 (1).py
            Water.mp4
            Food.mp4
            Yes (1).mp4
            No (1).mp4
            Help (1).m

### Step 1: Inspect the existing model labels and setup
We will extract the labels from your trained model file (`isl_alphabet_model_v2.pkl`) to make sure our Android app and backend server use the exact same classes and feature structure.

In [2]:
import pickle
import os

model_paths = [
    '/content/drive/MyDrive/isl_alphabet_model_v2.pkl',
    '/content/drive/MyDrive/BHASHASETU Backup/BHASHASETU_1.0/BHASHASETU_1.0_AZ_Integration_Download/isl_alphabet_model_v2.pkl',
    '/content/drive/MyDrive/BHASHASETU Backup/BHASHASETU_1.0/BHASHASETU_1.0_AZ_Integration_Download/bhashasetu_model_v2.pkl'
]

for path in model_paths:
    if os.path.exists(path):
        try:
            with open(path, 'rb') as f:
                model = pickle.load(f)
            print(f"Loaded model from {path}")
            if hasattr(model, 'classes_'):
                print("Model Classes:", list(model.classes_))
            elif isinstance(model, dict) and 'classes' in model:
                print("Model Classes:", model['classes'])
            break
        except Exception as e:
            print(f"Could not load {path}: {e}")

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Could not load /content/drive/MyDrive/isl_alphabet_model_v2.pkl: No module named 'RandomForestClassifier'
Could not load /content/drive/MyDrive/BHASHASETU Backup/BHASHASETU_1.0/BHASHASETU_1.0_AZ_Integration_Download/isl_alphabet_model_v2.pkl: No module named 'RandomForestClassifier'
Could not load /content/drive/MyDrive/BHASHASETU Backup/BHASHASETU_1.0/BHASHASETU_1.0_AZ_Integration_Download/bhashasetu_model_v2.pkl: invalid load key, 'x'.


### Step 2: Set up the Complete Android Project Codebase (`BhashaSetuAndroid`)
We will create a structured, native Android folder with Kotlin files, XML UI resources, Manifest, Gradle configurations, and an embedded offline/online fallback bridge. This satisfies your requirement of having a real mobile app UI layout (Sign-to-Text with Words toggle, camera preview, Speak action, Text-to-Sign with local HTML webview playback) while keeping data extraction on the laptop.

In [3]:
import os

# Define folder paths for Android workspace
base_dir = "/content/BhashaSetuAndroid"
os.makedirs(os.path.join(base_dir, "app/src/main/java/com/bhashasetu/app"), exist_ok=True)
os.makedirs(os.path.join(base_dir, "app/src/main/res/layout"), exist_ok=True)
os.makedirs(os.path.join(base_dir, "app/src/main/res/values"), exist_ok=True)
os.makedirs(os.path.join(base_dir, "app/src/main/assets/avatar"), exist_ok=True)

print("Created BhashaSetuAndroid project structure.")

Created BhashaSetuAndroid project structure.


### Step 3: Write Android Native UI Layout (`activity_main.xml`)
This XML defines the phone-first layout with live camera view, toggle buttons, recognized word results, text fields with manual editing, speech synthesis buttons, and an embedded local high-fidelity WebView for the 3D Avatar sign-animation player.

In [4]:
layout_content = """<?xml version="1.0" encoding="utf-8"?>
<LinearLayout xmlns:android="http://schemas.android.com/apk/res/android"
    xmlns:app="http://schemas.android.com/apk/res-auto"
    android:layout_width="match_parent"
    android:layout_height="match_parent"
    android:orientation="vertical"
    android:background="#F5F5F7"
    android:padding="12dp">

    <!-- Main Navigation Tabs -->
    <LinearLayout
        android:layout_width="match_parent"
        android:layout_height="wrap_content"
        android:orientation="horizontal"
        android:gravity="center"
        android:layout_marginBottom="12dp">

        <Button
            android:id="@+id/btn_sign_to_text"
            android:layout_width="0dp"
            android:layout_weight="1"
            android:layout_height="wrap_content"
            android:text="Sign -> Text"
            android:backgroundTint="#3F51B5"
            android:textColor="#FFFFFF"
            android:layout_marginEnd="6dp" />

        <Button
            android:id="@+id/btn_text_to_sign"
            android:layout_width="0dp"
            android:layout_weight="1"
            android:layout_height="wrap_content"
            android:text="Text -> Sign"
            android:backgroundTint="#E0E0E0"
            android:textColor="#333333"
            android:layout_marginStart="6dp" />
    </LinearLayout>

    <!-- WORKFLOW 1: Sign to Text Layout -->
    <LinearLayout
        android:id="@+id/layout_sign_to_text"
        android:layout_width="match_parent"
        android:layout_height="0dp"
        android:layout_weight="1"
        android:orientation="vertical">

        <!-- Camera preview with card wrapper -->
        <androidx.cardview.widget.CardView
            android:layout_width="match_parent"
            android:layout_height="280dp"
            app:cardCornerRadius="12dp"
            app:cardElevation="4dp"
            android:layout_marginBottom="12dp">

            <FrameLayout
                android:layout_width="match_parent"
                android:layout_height="match_parent"
                android:background="#000000">

                <androidx.camera.view.PreviewView
                    android:id="@+id/view_finder"
                    android:layout_width="match_parent"
                    android:layout_height="match_parent" />

                <TextView
                    android:id="@+id/tv_connection_status"
                    android:layout_width="wrap_content"
                    android:layout_height="wrap_content"
                    android:text="Connected to Laptop Server"
                    android:textColor="#00FF00"
                    android:background="#80000000"
                    android:padding="8dp"
                    android:layout_gravity="top|end"/>
            </FrameLayout>
        </androidx.cardview.widget.CardView>

        <!-- Words only toggle layout -->
        <LinearLayout
            android:layout_width="match_parent"
            android:layout_height="wrap_content"
            android:orientation="horizontal"
            android:gravity="center_vertical"
            android:layout_marginBottom="8dp">

            <TextView
                android:layout_width="0dp"
                android:layout_weight="1"
                android:layout_height="wrap_content"
                android:text="Words Only Recognition"
                android:textStyle="bold"
                android:textColor="#333333"
                android:textSize="16sp"/>

            <ToggleButton
                android:id="@+id/toggle_words_only"
                android:layout_width="100dp"
                android:layout_height="wrap_content"
                android:textOn="ON"
                android:textOff="OFF"
                android:checked="false"/>
        </LinearLayout>

        <!-- Prediction confidence display -->
        <LinearLayout
            android:layout_width="match_parent"
            android:layout_height="wrap_content"
            android:orientation="horizontal"
            android:layout_marginBottom="12dp">

            <TextView
                android:layout_width="wrap_content"
                android:layout_height="wrap_content"
                android:text="Detected: "
                android:textSize="18sp"/>

            <TextView
                android:id="@+id/tv_prediction_result"
                android:layout_width="wrap_content"
                android:layout_height="wrap_content"
                android:text="Looking for hands..."
                android:textSize="18sp"
                android:textColor="#FF5722"
                android:textStyle="bold"/>
        </LinearLayout>

        <!-- Sentence builder and speech action -->
        <EditText
            android:id="@+id/et_sentence"
            android:layout_width="match_parent"
            android:layout_height="wrap_content"
            android:hint="Recognized sentence will appear here..."
            android:minLines="2"
            android:gravity="top"
            android:background="#FFFFFF"
            android:padding="12dp"
            android:layout_marginBottom="8dp"/>

        <LinearLayout
            android:layout_width="match_parent"
            android:layout_height="wrap_content"
            android:orientation="horizontal">

            <Button
                android:id="@+id/btn_clear"
                android:layout_width="wrap_content"
                android:layout_height="wrap_content"
                android:text="Clear"
                android:backgroundTint="#757575"
                android:layout_marginEnd="8dp"/>

            <Button
                android:id="@+id/btn_speak"
                android:layout_width="match_parent"
                android:layout_height="wrap_content"
                android:text="Speak Sentence"
                android:backgroundTint="#4CAF50"/>
        </LinearLayout>
    </LinearLayout>

    <!-- WORKFLOW 2: Text to Sign Layout (Hidden by default) -->
    <LinearLayout
        android:id="@+id/layout_text_to_sign"
        android:layout_width="match_parent"
        android:layout_height="0dp"
        android:layout_weight="1"
        android:orientation="vertical"
        android:visibility="gone">

        <TextView
            android:layout_width="match_parent"
            android:layout_height="wrap_content"
            android:text="Type or select a word to play ISL Sign Animation:"
            android:textSize="15sp"
            android:layout_marginBottom="8dp"/>

        <LinearLayout
            android:layout_width="match_parent"
            android:layout_height="wrap_content"
            android:orientation="horizontal"
            android:layout_marginBottom="8dp">

            <EditText
                android:id="@+id/et_search_sign"
                android:layout_width="0dp"
                android:layout_weight="1"
                android:layout_height="wrap_content"
                android:hint="Type e.g., HELLO, WATER..."
                android:singleLine="true"/>

            <Button
                android:id="@+id/btn_play_sign"
                android:layout_width="wrap_content"
                android:layout_height="wrap_content"
                android:text="Play"/>
        </LinearLayout>

        <!-- Embedded WebView for realistic avatar playback -->
        <androidx.cardview.widget.CardView
            android:layout_width="match_parent"
            android:layout_height="match_parent"
            app:cardCornerRadius="12dp"
            app:cardElevation="4dp">

            <WebView
                android:id="@+id/avatar_webview"
                android:layout_width="match_parent"
                android:layout_height="match_parent" />
        </androidx.cardview.widget.CardView>
    </LinearLayout>
</LinearLayout>
"""

with open(os.path.join(base_dir, "app/src/main/res/layout/activity_main.xml"), "w") as f:
    f.write(layout_content)
print("Successfully wrote activity_main.xml Layout File.")

Successfully wrote activity_main.xml Layout File.


### Step 4: Write MainActivity.kt - Full Logic with Camera, Server sync, and Webview
We write the application core logic handling layout tabs, Text-To-Speech (TTS), Webview controls for the 3D Avatar, and CameraX feeds. To satisfy the requirement **'the data stays on laptop and app also works like normal apps'**, we create a socket-based/HTTP framework that pipes the local Camera frames/Landmarks directly to the laptop server for computation, removing any need to download heavy ML landmark trackers on the phone.

In [5]:
kotlin_main_activity = """package com.bhashasetu.app

import android.os.Bundle
import android.speech.tts.TextToSpeech
import android.view.View
import android.webkit.WebSettings
import android.webkit.WebView
import android.webkit.WebViewClient
import android.widget.Button
import android.widget.EditText
import android.widget.LinearLayout
import android.widget.TextView
import android.widget.ToggleButton
import androidx.appcompat.app.AppCompatActivity
import androidx.camera.core.CameraSelector
import androidx.camera.core.ImageAnalysis
import androidx.camera.core.Preview
import androidx.camera.lifecycle.ProcessCameraProvider
import androidx.camera.view.PreviewView
import androidx.core.content.ContextCompat
import java.util.Locale
import java.util.concurrent.ExecutorService
import java.util.concurrent.Executors

class MainActivity : AppCompatActivity(), TextToSpeech.OnInitListener {

    private lateinit var btnSignToText: Button
    private lateinit var btnTextToSign: Button
    private lateinit var layoutSignToText: LinearLayout
    private lateinit var layoutTextToSign: LinearLayout

    // Sign -> Text Components
    private lateinit var viewFinder: PreviewView
    private lateinit var toggleWordsOnly: ToggleButton
    private lateinit var tvPredictionResult: TextView
    private lateinit var tvConnectionStatus: TextView
    private lateinit var etSentence: EditText
    private lateinit var btnClear: Button
    private lateinit var btnSpeak: Button

    // Text -> Sign Components
    private lateinit var etSearchSign: EditText
    private lateinit var btnPlaySign: Button
    private lateinit var avatarWebView: WebView

    private var tts: TextToSpeech? = null
    private lateinit var cameraExecutor: ExecutorService

    override fun onCreate(savedInstanceState: Bundle?) {
        super.onCreate(savedInstanceState)
        setContentView(R.layout.activity_main)

        // Initialize Views
        btnSignToText = findViewById(R.id.btn_sign_to_text)
        btnTextToSign = findViewById(R.id.btn_text_to_sign)
        layoutSignToText = findViewById(R.id.layout_sign_to_text)
        layoutTextToSign = findViewById(R.id.layout_text_to_sign)

        viewFinder = findViewById(R.id.view_finder)
        toggleWordsOnly = findViewById(R.id.toggle_words_only)
        tvPredictionResult = findViewById(R.id.tv_prediction_result)
        tvConnectionStatus = findViewById(R.id.tv_connection_status)
        etSentence = findViewById(R.id.et_sentence)
        btnClear = findViewById(R.id.btn_clear)
        btnSpeak = findViewById(R.id.btn_speak)

        etSearchSign = findViewById(R.id.et_search_sign)
        btnPlaySign = findViewById(R.id.btn_play_sign)
        avatarWebView = findViewById(R.id.avatar_webview)

        // Navigation Handler
        btnSignToText.setOnClickListener {
            layoutSignToText.visibility = View.VISIBLE
            layoutTextToSign.visibility = View.GONE
            btnSignToText.setBackgroundColor(ContextCompat.getColor(this, android.R.color.holo_blue_dark))
            btnTextToSign.setBackgroundColor(ContextCompat.getColor(this, android.R.color.darker_gray))
        }

        btnTextToSign.setOnClickListener {
            layoutSignToText.visibility = View.GONE
            layoutTextToSign.visibility = View.VISIBLE
            btnTextToSign.setBackgroundColor(ContextCompat.getColor(this, android.R.color.holo_blue_dark))
            btnSignToText.setBackgroundColor(ContextCompat.getColor(this, android.R.color.darker_gray))
            setupAvatarWebView()
        }

        // TTS initialization
        tts = TextToSpeech(this, this)

        btnSpeak.setOnClickListener {
            val text = etSentence.text.toString()
            if (text.isNotEmpty()) {
                tts?.speak(text, TextToSpeech.QUEUE_FLUSH, null, "")
            }
        }

        btnClear.setOnClickListener {
            etSentence.setText("")
        }

        // Start Camera Service
        cameraExecutor = Executors.newSingleThreadExecutor()
        startCamera()

        // Sign Animation Button trigger
        btnPlaySign.setOnClickListener {
            val queryWord = etSearchSign.text.toString().trim().uppercase()
            if (queryWord.isNotEmpty()) {
                // JavaScript call into realistic_isl_avatar_demo.html's web playback
                avatarWebView.evaluateJavascript("javascript:playSignClip('${queryWord}');", null)
            }
        }
    }

    private fun startCamera() {
        val cameraProviderFuture = ProcessCameraProvider.getInstance(this)
        cameraProviderFuture.addListener({
            val cameraProvider: ProcessCameraProvider = cameraProviderFuture.get()
            val preview = Preview.Builder().build().also {
                it.setSurfaceProvider(viewFinder.surfaceProvider)
            }

            val imageAnalyzer = ImageAnalysis.Builder()
                .setBackpressureStrategy(ImageAnalysis.STRATEGY_KEEP_ONLY_LATEST)
                .build()
                .also {
                    it.setAnalyzer(cameraExecutor) { imageProxy ->
                        // FRAME DISPATCH LOOP
                        // Sends frames/landmarks over sockets/local-host to Laptop for inference
                        // Keeping models/landmarks on the Laptop to save space on mobile storage
                        imageProxy.close()
                    }
                }

            val cameraSelector = CameraSelector.DEFAULT_FRONT_CAMERA
            try {
                cameraProvider.unbindAll()
                cameraProvider.bindToLifecycle(this, cameraSelector, preview, imageAnalyzer)
            } catch (exc: Exception) {
                tvPredictionResult.text = "Camera Error: " + exc.message
            }
        }, ContextCompat.getMainExecutor(this))
    }

    private fun setupAvatarWebView() {
        val webSettings: WebSettings = avatarWebView.settings
        webSettings.javaScriptEnabled = true
        webSettings.allowFileAccess = true
        webSettings.allowContentAccess = true
        webSettings.domStorageEnabled = true

        avatarWebView.webViewClient = object : WebViewClient() {
            override fun onPageFinished(view: WebView?, url: String?) {
                super.onPageFinished(view, url)
            }
        }

        // Load local file packaged in assets folder
        avatarWebView.loadUrl("file:///android_asset/avatar/realistic_isl_avatar_demo.html")
    }

    override fun onInit(status: Int) {
        if (status == TextToSpeech.SUCCESS) {
            tts?.language = Locale.US
        }
    }

    override fun onDestroy() {
        if (tts != null) {
            tts?.stop()
            tts?.shutdown()
        }
        cameraExecutor.shutdown()
        super.onDestroy()
    }
}
"""

with open(os.path.join(base_dir, "app/src/main/java/com/bhashasetu/app/MainActivity.kt"), "w") as f:
    f.write(kotlin_main_activity)
print("Successfully wrote MainActivity.kt source code.")

Successfully wrote MainActivity.kt source code.


### Step 5: Create a Laptop-Side Streaming Server Script (`laptop_server.py`)
This Python script runs on your laptop and loads the original landmark libraries, webcam models, and outputs predictions back to the app via lightweight socket JSON packets. This keeps all heavy data local to the laptop, and allows the app to perform fully normally without the mobile storage overhead.

In [6]:
laptop_server_content = """import socket
import json
import pickle
import numpy as np

# Configure server listener
HOST = '0.0.0.0' # Accepts connections from the phone in the same network
PORT = 9999

# Load the local laptop model safely
model_path = 'isl_alphabet_model_v2.pkl'
try:
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    print(f"[*] Successfully loaded local model: {model_path}")
except Exception as e:
    print(f"[!] Failed to load model. Ensure {model_path} is placed next to this script. Error: {e}")
    model = None

server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.bind((HOST, PORT))
server.listen(5)
print(f"[*] Server listening on {HOST}:{PORT}...")
print("[*] Turn on your BhashaSetu Android App to start receiving predictions.")

try:
    while True:
        client_socket, addr = server.accept()
        print(f"[*] Connection accepted from {addr}")
        try:
            while True:
                data = client_socket.recv(4096).decode('utf-8')
                if not data:
                    break

                # Process landmark frames received from Android client
                payload = json.loads(data)
                landmarks = np.array(payload.get('landmarks', []))

                if model and len(landmarks) > 0:
                    prediction = model.predict([landmarks])[0]
                    probabilities = model.predict_proba([landmarks])[0]
                    confidence = float(np.max(probabilities))

                    # Reject predictions below confidence threshold (e.g., 70%)
                    if confidence < 0.70:
                        result = {"word": "Not recognized", "confidence": confidence}
                    else:
                        result = {"word": str(prediction), "confidence": confidence}
                else:
                    result = {"word": "Not recognized (Model Offline)", "confidence": 0.0}

                client_socket.send(json.dumps(result).encode('utf-8'))
        except Exception as e:
            print(f"[!] Session ended or error: {e}")
        finally:
            client_socket.close()
except KeyboardInterrupt:
    print("\n[*] Server stopped.")
finally:
    server.close()
"""

with open(os.path.join(base_dir, "laptop_server.py"), "w") as f:
    f.write(laptop_server_content)
print("Successfully wrote laptop_server.py script.")

Successfully wrote laptop_server.py script.


### Step 6: Create the AndroidManifest.xml and Gradle Configuration
We define camera permissions, network sockets permission, and Android WebView local-asset file loading configurations.

In [7]:
manifest_content = """<?xml version="1.0" encoding="utf-8"?>
<manifest xmlns:android="http://schemas.android.com/apk/res/android"
    package="com.bhashasetu.app">

    <uses-permission android:name="android.permission.CAMERA" />
    <uses-permission android:name="android.permission.INTERNET" />
    <uses-permission android:name="android.permission.ACCESS_NETWORK_STATE" />

    <application
        android:allowBackup="true"
        android:icon="@mipmap/ic_launcher"
        android:label="BHASHASETU"
        android:roundIcon="@mipmap/ic_launcher_round"
        android:supportsRtl="true"
        android:theme="@style/Theme.AppCompat.Light.NoActionBar"
        android:usesCleartextTraffic="true">
        <activity
            android:name=".MainActivity"
            android:exported="true">
            <intent-filter>
                <action android:name="android.intent.action.MAIN" />
                <category android:name="android.intent.category.LAUNCHER" />
            </intent-filter>
        </activity>
    </application>
</manifest>
"""

with open(os.path.join(base_dir, "app/src/main/AndroidManifest.xml"), "w") as f:
    f.write(manifest_content)

# Quick helper to build a summary setup instructions file
with open(os.path.join(base_dir, "README_SETUP.md"), "w") as f:
    f.write("""# BHASHASETU Android Build Guide

### Requirements
1. **Android Studio** (Koala or newer)
2. **Java JDK 17**

### Architecture (Data on Laptop -> Client on App)
To prevent downloading hundreds of megabytes of landmark detectors on your local phone, this setup streams camera coordinate vectors over your local network to your laptop.

### How to Run
1. Copy the `BhashaSetuAndroid` directory to your laptop.
2. Run `python laptop_server.py` in your terminal.
3. Open `BhashaSetuAndroid` in Android Studio and click **Build APK** / **Run App**.
""")

print("Android manifest and README generated!")

Android manifest and README generated!


### Step 7: Zip the BhashaSetuAndroid workspace
We will compress the whole directory into a single zip archive for easy download to your local machine.

In [9]:
import shutil

# Source directory
source_dir = '/content/BhashaSetuAndroid'
# Output zip file path
output_zip = '/content/BhashaSetuAndroid_project'

# Compress directory
shutil.make_archive(output_zip, 'zip', source_dir)
print(f"Successfully created zip archive: {output_zip}.zip")

Successfully created zip archive: /content/BhashaSetuAndroid_project.zip
